# Deploy appmux from a release zip, inside Databricks

This notebook takes the release zip (`appmux-<version>.zip`) you uploaded to a Unity Catalog
volume or a workspace folder and sets up the whole slot from the workspace itself, with no
Databricks CLI and no local machine ([design 13.1](https://github.com/timezlab/databricks-appmux/blob/main/docs/product/v1-design.md#131-what-the-operator-sets-up)).

**What you fill in.** Only the Settings cell (cell 1), and only `ZIP_PATH` is required. The
rest depends on your setup and may stay empty; they come in the order of the cell:

| Variable | Set it when |
|----------|-------------|
| `ZIP_PATH` | Always: where the release zip is. |
| `FIRST_ADMIN_EMAIL`, `MORE_PLATFORM_ADMIN_EMAILS` | Someone other than you, or more than you, administers appmux. |
| `APP_ACL` | Anyone besides you and the workspace admins may open appmux. |
| `CONTENT_STORE_FOLDER` | You may delete and create the App again. |
| `SP_GRANTS` | A child app needs the App's service principal to read something (a secret scope, a folder, a volume, a warehouse). Not for appmux's own needs: the notebook makes the Lakebase project and roles itself. |
| `REPLACE_EXISTING_FOLDER` | You run the notebook again with the same deploy folder (an upgrade, or a new App). |
| `EXPECTED_SHA256` | No `.sha256` file sits beside the zip. |

**Run the cells once, from top to bottom.** The order matters:

| # | Cell | Why it sits here |
|---|------|------------------|
| 1 | Settings | The only cell you edit. |
| 2 | Connect | Everything below talks to the workspace as you. |
| 3 | Unzip | Checks the hash, then unpacks into a workspace folder (an App deploys only from one). |
| 4 | `app.yaml` | The first admin, and the content store folder if you chose one: the zip carries placeholders. |
| 5 | Lakebase | The App's database resource names a Lakebase database, so Lakebase comes before the App. |
| 6 | The App | Created with user authorization (`user_api_scopes`) and the Lakebase resource. |
| 7 | Permissions | 7a: who may open appmux. 7b: what appmux's service principal may use (the content store folder and your `SP_GRANTS`). Both need the App, so they come after cell 6, and 7b must come before the deploy. |
| 8 | Database driver | `psycopg` from the zip's own `wheels/`, hash-checked. |
| 9 | Postgres roles | They need the App's service principal, which exists only after cell 6. |
| 10 | Load the schema | Needs the roles of cell 9. Runs with **your** login, never the App's. |
| 11 | Deploy | Only now is the registry ready for the App's first start. |
| 12 | Restart if scopes changed | A changed scope reaches tokens only after a stop and start. |
| 13 | Status | URL and what to check next. |

Running a cell again is safe: each step looks at what exists before it creates anything.

**If you delete the App and create it again.** The new App has a new service principal, so
it has none of the old one's access. Run this notebook from the top with the same settings and
`REPLACE_EXISTING_FOLDER = True` (the deploy folder is still there); it gives the new service
principal what `APP_ACL`, `CONTENT_STORE_FOLDER` and `SP_GRANTS` list. appmux then notices the change at its first start and **stops every running app**: it
cannot know what each app reads at run time, and an app that holds the shared token would
pass its health check and then fail with 403s nobody sees. Each app's admin gives the new
service principal (its application id is named in the notice on the app's Overview page) what that app needs,
then presses **Start**. appmux grants nothing for the apps. This needs a release newer than
v0.1.0; v0.1.0 does not look for the change. The registry (Lakebase) must still exist.

**Before you start**

- The zip and its `.sha256` file are in the same folder, in a volume (`/Volumes/...`) or a
  workspace folder (`/Workspace/...`). The notebook refuses a zip it cannot verify.
- You may create Apps and Lakebase projects in this workspace. You become the Lakebase
  project owner, which is what the roles and the schema need.
- A workspace admin has switched on **user authorization** for Databricks Apps (a workspace
  setting, once). Cell 6 asks for the scopes, but cannot switch the feature on.
- Compute: its Python needs `pip`, and **Python 3.11** is the one the zip's `psycopg-binary`
  wheel is built for (on another version cell 8 falls back to the package index). The
  notebook installs and pins no Databricks SDK, so it uses whichever one your compute has.

**What was and was not tested.** Every code cell was run, in order, on Python 3.11 with
`databricks-sdk` 0.33.0 and with 0.161.0, against a fake workspace API (so the request
shapes come from the SDK's own definitions and the CLI's help, not from a live workspace) and
a real PostgreSQL 17 with TLS, with the **real v0.1.0 zip**: the hash check, the unpack, the
`app.yaml` edit, the offline wheel install, the group roles and grants, and the real schema
loader (exit 0, then 3 on a second pass). The refusals were run too: wrong or missing hash,
a zip entry outside the folder, a symbolic link, a folder that already holds files.
The content store folder needs a release newer than v0.1.0, which does not read the setting
and has no template for it in `app.yaml` (cell 4 then stops). The folder cells were run with
the v0.1.0 zip whose `app.yaml` was replaced by the one in the repository; the refusal was
run with the untouched v0.1.0 zip.
The notebook has **not** been run on a real workspace: Lakebase's real response fields, the
platform's timing for the service principal role, serverless compute, a notebook's own
token for `generate-database-credential`, and the calls of cells 7a and 7b (the App's
permissions, secret ACLs, folder and Unity Catalog grants) are unconfirmed. If a cell stops with an API error,
the message names the call; send it back.

## 1. Settings

This is the only cell you edit. Block A is required (`ZIP_PATH`). Block B holds what depends on
your setup, and every value there may stay empty. Blocks C and D have working defaults.

In [ ]:
# Fill in block A. Fill in block B only where it applies to you: every value there may stay
# empty. Blocks C and D have working defaults.

# ================================================================================================
# A. Required: fill this in
# ================================================================================================

# Where the release is: a Unity Catalog volume path or a workspace path. The matching
# <zip>.sha256 file must sit beside it.
ZIP_PATH = "/Volumes/<catalog>/<schema>/<volume>/appmux-0.1.0.zip"

# ================================================================================================
# B. Fill in what applies to you
# ================================================================================================

# Platform admins. Empty first admin: the person running this notebook. Each address gets a
# Postgres role in appmux_platform_writers (cell 9) and is listed in app.yaml (cell 4).
FIRST_ADMIN_EMAIL = ""
MORE_PLATFORM_ADMIN_EMAILS = []

# Who may open appmux: the App's own permissions. They belong to the App object, so deleting
# the App deletes them: list everyone here and run cell 7a again after creating it anew.
# Each entry names one of user_name, group_name or service_principal_name (the application
# id), and a permission_level of CAN_USE or CAN_MANAGE. Empty: only you and workspace admins
# can open it. appmux's own grants decide which child each person reaches.
APP_ACL = [
    # {"group_name": "users", "permission_level": "CAN_USE"},  # everyone in the workspace
    # {"user_name": "dana@example.com", "permission_level": "CAN_USE"},
]

# Where appmux keeps the files it stores (release snapshots and wheels). It needs the
# folder's path to stay the same when the App is deleted and created again, and the new App
# gets a new service principal, so the default (a folder of the service principal itself)
# would be lost. A fixed folder is not: cell 7b creates it and gives the service principal
# CAN_MANAGE on it. Use a folder only appmux writes to: it deletes the files it no longer
# needs. Not inside DEPLOY_FOLDER. Empty: the service principal's own folder.
CONTENT_STORE_FOLDER = ""  # for example "/Workspace/Shared/appmux-store"

# What the child apps need the App's service principal to read (their secret scopes, a folder,
# a volume, a warehouse). Leave it empty if none do. What appmux itself needs is not listed
# here: this notebook already makes the Lakebase project and the Postgres roles (cells 5 and
# 9), and cell 7b gives the service principal the CONTENT_STORE_FOLDER above. Cell 7b runs
# these every time, so a new service principal gets the same access as the old one. Each entry:
#   {"type": "secret_scope", "scope": "team-secrets", "permission": "READ"}
#   {"type": "workspace_folder", "path": "/Workspace/Shared/files",
#    "permission_level": "CAN_READ"}
#   {"type": "unity_catalog", "securable_type": "volume", "full_name": "main.apps.files",
#    "privileges": ["READ_VOLUME"]}
#   {"type": "permission", "object_type": "warehouses", "object_id": "<id>",
#    "permission_level": "CAN_USE"}
# You need the right to grant each one (for a secret scope: MANAGE on it). appmux never
# grants on its own: a child's secret references are read with the service principal.
SP_GRANTS = []

# Set True when you run this notebook again with the same DEPLOY_FOLDER: an upgrade, or a
# new App after the old one was deleted. A non-empty DEPLOY_FOLDER is refused unless this is
# True (an upgrade replaces the folder, so files of the old release do not linger).
REPLACE_EXISTING_FOLDER = False

# Used only when there is no .sha256 file beside the zip (copy it from the release page).
EXPECTED_SHA256 = ""

# ================================================================================================
# C. Defaults: change them only with a reason
# ================================================================================================

APP_NAME = "appmux"
# Where the zip is unpacked and deployed from. Must be under /Workspace.
# Empty: /Workspace/Users/<you>/<APP_NAME>.
DEPLOY_FOLDER = ""
LAKEBASE_PROJECT_ID = "appmux"  # lowercase letters, digits and hyphens
LAKEBASE_DATABASE = "databricks_postgres"  # a new project's default database

# The sizes the API accepts are MEDIUM, LARGE and XLARGE. appmux reads its memory limit from
# the container at start, so it follows the size: measured on a Free Edition workspace, MEDIUM
# gave 8192 MB, LARGE 12288 MB and XLARGE 49152 MB. Changing the size of an App that runs
# restarts it. None: leave the size as the platform made it.
COMPUTE_SIZE = "MEDIUM"

# How long to wait for slow platform steps, in seconds. Raise them if a cell times out.
WAIT_LAKEBASE_S = 900
WAIT_APP_S = 1200

# ================================================================================================
# D. Do not change
# ================================================================================================

# Design 13.1 item 4: the attach checks and the Add resource pickers read as the visitor with
# these scopes; one missing, and that type is unverified or listed with appmux's own access.
# Do not shorten this list.
USER_API_SCOPES = [
    "iam.current-user:read",
    "workspace.workspace:read",
    "sql",
    "files.files:read",
    "catalog.tables:read",
    "catalog.connections:read",
    "dashboards.genie:read",
    "serving.serving-endpoints:read",
    "catalog.catalogs:read",
    "catalog.schemas:read",
    "vectorsearch.vector-search-indexes:read",
    "apps",
    "postgres",
]

## 2. Connect and check the settings

The notebook calls the workspace REST API through `WorkspaceClient().api_client.do`, which
every Databricks SDK version has, so it does not depend on the SDK version of your compute
(the typed Lakebase and Apps methods are newer than many runtimes).

In [ ]:
import contextlib
import hashlib
import importlib
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound

# What the settings must look like; they are checked below, before anything is created.
EMAIL = re.compile(r"^[^@\s,:#]+@[^@\s,:#]+$")
ID = re.compile(r"^[a-z][a-z0-9-]{0,62}$")
# The keys an SP_GRANTS entry of each type takes: (required, optional).
GRANT_KEYS = {
    "secret_scope": ({"scope"}, {"permission"}),
    "workspace_folder": ({"path"}, {"permission_level"}),
    "unity_catalog": ({"securable_type", "full_name", "privileges"}, set()),
    "permission": ({"object_type", "object_id", "permission_level"}, set()),
}

w = WorkspaceClient()
me = w.current_user.me().user_name
print(f"Running as {me} on Python {sys.version.split()[0]}")

if "<" in ZIP_PATH or ">" in ZIP_PATH:
    raise ValueError("Cell 1: set ZIP_PATH to the real path of the release zip.")
if not ID.match(LAKEBASE_PROJECT_ID):
    raise ValueError("Cell 1: LAKEBASE_PROJECT_ID takes lowercase letters, digits, hyphens.")
deploy_folder = Path(DEPLOY_FOLDER or f"/Workspace/Users/{me}/{APP_NAME}")
if deploy_folder.parts[:2] != ("/", "Workspace"):
    raise ValueError("Cell 1: DEPLOY_FOLDER must be under /Workspace; an App deploys from there.")
first_admin = (FIRST_ADMIN_EMAIL or me).lower()
platform_admins = list(
    dict.fromkeys([first_admin, *[a.lower() for a in MORE_PLATFORM_ADMIN_EMAILS]])
)
for address in platform_admins:
    if not EMAIL.match(address):
        raise ValueError(f"Cell 1: {address!r} is not an e-mail address.")
print(f"Deploy folder: {deploy_folder}")
print(f"Platform admins: {', '.join(platform_admins)}")

# The content store folder: plain characters only (it goes into app.yaml), under /Workspace/,
# and apart from the deploy folder (cell 3 empties that folder; appmux deletes files in this one).
store_folder = CONTENT_STORE_FOLDER.strip().rstrip("/")
if store_folder:
    store_path = Path(store_folder)
    if not re.fullmatch(r"/Workspace/[A-Za-z0-9._@+-]+(/[A-Za-z0-9._@+-]+)*", store_folder) or {
        ".",
        "..",
    } & set(store_path.parts):
        raise ValueError(
            "Cell 1: CONTENT_STORE_FOLDER must be a path under /Workspace/ "
            "of letters, digits and . _ @ + - only."
        )
    if (
        store_path == deploy_folder
        or store_path in deploy_folder.parents
        or deploy_folder in store_path.parents
    ):
        raise ValueError(
            "Cell 1: CONTENT_STORE_FOLDER and DEPLOY_FOLDER must be two separate folders."
        )
    print(f"Content store folder: {store_folder}")
else:
    print("Content store folder: the service principal's own (lost if the App is deleted)")

# Check the permission lists now, so a typo does not surface after the App was created.
if not all(isinstance(entry, dict) for entry in [*APP_ACL, *SP_GRANTS]):
    raise ValueError("Cell 1: APP_ACL and SP_GRANTS hold dictionaries.")
for entry in APP_ACL:
    names = {"user_name", "group_name", "service_principal_name"} & set(entry)
    if (
        len(names) != 1
        or entry.get("permission_level") not in ("CAN_USE", "CAN_MANAGE")
        or len(entry) != 2
    ):
        raise ValueError(
            f"Cell 1: bad APP_ACL entry {entry}: one principal "
            "and a permission_level of CAN_USE or CAN_MANAGE."
        )
for entry in SP_GRANTS:
    required, optional = GRANT_KEYS.get(entry.get("type"), (None, None))
    keys = set(entry) - {"type"}
    if required is None or not required <= keys or keys - required - optional:
        raise ValueError(f"Cell 1: bad SP_GRANTS entry {entry}; see the examples in cell 1.")


def api(method, path, *, query=None, body=None):
    """One REST call; returns the parsed JSON ({} when there is no body)."""
    return w.api_client.do(method, path, query=query, body=body) or {}


def get_or_none(path, query=None):
    try:
        return api("GET", path, query=query)
    except NotFound:
        return None


def list_all(path, key):
    """Every item of a paginated list call."""
    items, token = [], None
    while True:
        page = api("GET", path, query={"page_token": token} if token else None)
        items += page.get(key, [])
        token = page.get("next_page_token")
        if not token:
            return items


def wait_for(what, check, *, timeout_s, every_s=10):
    """Call check() until it returns something other than None."""
    deadline = time.monotonic() + timeout_s
    while True:
        result = check()
        if result is not None:
            return result
        if time.monotonic() > deadline:
            raise TimeoutError(f"Timed out after {timeout_s}s waiting for {what}.")
        time.sleep(every_s)


def wait_operation(operation, what):
    """Wait for a Lakebase long-running operation and raise if it failed."""

    def done():
        current = api("GET", f"/api/2.0/postgres/{operation['name']}")
        return current if current.get("done") else None

    finished = wait_for(what, done, timeout_s=WAIT_LAKEBASE_S, every_s=5)
    if finished.get("error"):
        raise RuntimeError(f"{what} failed: {finished['error']}")
    return finished

## 3. Check the hash and unzip

The zip's top folder (`appmux-<version>/`) is dropped, so `app.yaml` ends up at the root of
the deploy folder, which is what the App reads. Entries that would land outside the folder,
and symbolic links, are refused.

In [ ]:
zip_path = Path(ZIP_PATH)
sha_path = Path(str(zip_path) + ".sha256")
# What a release must contain; checked after the unpack (3d).
REQUIRED_IN_RELEASE = [
    "app.yaml",
    "requirements.txt",
    "appmux/__main__.py",
    "migrations",
    "web/out",
    "wheels",
]

# 3a. The hash. No hash to compare with means no deploy.
if sha_path.exists():
    expected_sha256 = sha_path.read_text().split()[0].lower()
elif EXPECTED_SHA256:
    expected_sha256 = EXPECTED_SHA256.strip().lower()
else:
    raise FileNotFoundError(
        f"No {sha_path.name} beside the zip and EXPECTED_SHA256 is empty. "
        "Upload the .sha256 file from the same release, or paste its hash in cell 1."
    )
digest = hashlib.sha256()
with zip_path.open("rb") as zip_file:
    for chunk in iter(lambda: zip_file.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest() != expected_sha256:
    raise RuntimeError(
        f"{zip_path.name} does not match its SHA-256. Download it again from the GitHub "
        "Release and do not deploy this copy."
    )
print(f"SHA-256 matches for {zip_path.name}")

# 3b. The target folder: never overwrite what is there without being told to.
if deploy_folder.exists() and any(deploy_folder.iterdir()):
    if not REPLACE_EXISTING_FOLDER:
        count = sum(1 for p in deploy_folder.rglob("*") if p.is_file())
        raise FileExistsError(
            f"{deploy_folder} already holds {count} files. Set REPLACE_EXISTING_FOLDER = True "
            "in cell 1 to replace them with this release."
        )
    shutil.rmtree(deploy_folder)
deploy_folder.mkdir(parents=True, exist_ok=True)
root = deploy_folder.resolve()

# 3c. Unpack, keeping every file inside the folder.
with zipfile.ZipFile(zip_path) as archive:
    infos = archive.infolist()
    tops = {info.filename.split("/")[0] for info in infos}
    strip = f"{tops.pop()}/" if len(tops) == 1 else ""
    written = 0
    for info in infos:
        relative = info.filename[len(strip) :]
        if not relative:
            continue
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError(f"Refusing a symbolic link in the zip: {info.filename}")
        destination = (root / relative).resolve()
        if destination != root and root not in destination.parents:
            raise RuntimeError(f"Refusing a zip entry outside the folder: {info.filename}")
        if info.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(info) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)
        written += 1

# 3d. It must look like a release.
missing = [name for name in REQUIRED_IN_RELEASE if not (root / name).exists()]
if missing:
    raise RuntimeError(f"The zip is not an appmux release; missing: {', '.join(missing)}")
# A workspace folder may not keep the executable bit; appmux also restores it at start.
with contextlib.suppress(OSError):
    (root / "bin" / "slirp4netns").chmod(0o755)
version = re.search(r'__version__ = "([^"]+)"', (root / "appmux" / "__init__.py").read_text())
print(f"Unpacked appmux {version.group(1) if version else '?'}: {written} files in {root}")

## 4. Fill in `app.yaml`

The zip carries a placeholder address in `APPMUX_BOOTSTRAP_ADMINS`. The list is honoured only
while the registry's `platform_admins` setting is unset, so it decides who can sign in first.

If you set `CONTENT_STORE_FOLDER`, the commented `APPMUX_CONTENT_STORE_FOLDER` template of
`app.yaml` becomes an entry with that path. A release that has no such template is too old to
read the setting, and the cell stops instead of deploying one that would ignore it.

In [ ]:
app_yaml = root / "app.yaml"
text = app_yaml.read_text()
pattern = re.compile(r"(- name: APPMUX_BOOTSTRAP_ADMINS\s*\n\s*value:[ \t]*)\S+")
store_entry = re.compile(
    r"^([ \t]*)(?:# ?)?- name: APPMUX_CONTENT_STORE_FOLDER\n[ \t]*#?[ \t]*value:[^\n]*", re.M
)
if not pattern.search(text):
    raise RuntimeError("app.yaml has no APPMUX_BOOTSTRAP_ADMINS entry; this is not a v1 release.")
text = pattern.sub(lambda match: match.group(1) + ",".join(platform_admins), text, count=1)
if "example.com" in text:
    raise RuntimeError("app.yaml still holds the placeholder address.")
if store_folder:
    if not store_entry.search(text):
        raise RuntimeError(
            "app.yaml has no APPMUX_CONTENT_STORE_FOLDER entry: this release is older than the one "
            "that reads it. Use a newer zip, or leave CONTENT_STORE_FOLDER empty."
        )
    text = store_entry.sub(
        lambda match: (
            f"{match.group(1)}- name: APPMUX_CONTENT_STORE_FOLDER\n"
            f"{match.group(1)}  value: {store_folder}"
        ),
        text,
        count=1,
    )
app_yaml.write_text(text)
print(re.search(r"- name: APPMUX_BOOTSTRAP_ADMINS\s*\n\s*value:.*", text).group(0))
if store_folder:
    print(re.search(r"^[ \t]*- name: APPMUX_CONTENT_STORE_FOLDER\n.*", text, re.M).group(0))
else:
    print(
        "No content store folder set: appmux keeps its store in its service principal's own folder."
    )

## 5. Lakebase

Creates the project if it is not there (a new project comes with a default branch, a
read-write endpoint and the `databricks_postgres` database), then looks up the three names
the later cells need.

In [ ]:
project_name = f"projects/{LAKEBASE_PROJECT_ID}"
if get_or_none(f"/api/2.0/postgres/{project_name}") is None:
    operation = api(
        "POST",
        "/api/2.0/postgres/projects",
        query={"project_id": LAKEBASE_PROJECT_ID},
        body={"spec": {"display_name": LAKEBASE_PROJECT_ID}},
    )
    wait_operation(operation, f"creating Lakebase project {LAKEBASE_PROJECT_ID}")
    print(f"Created Lakebase project {LAKEBASE_PROJECT_ID}")
else:
    print(f"Lakebase project {LAKEBASE_PROJECT_ID} already exists; using it")


def default_branch():
    branches = list_all(f"/api/2.0/postgres/{project_name}/branches", "branches")
    defaults = [b for b in branches if (b.get("status") or {}).get("default")]
    if len(defaults) == 1:
        return defaults[0]["name"]
    return None


def read_write_endpoint():
    endpoints = list_all(f"/api/2.0/postgres/{branch_name}/endpoints", "endpoints")
    for endpoint in endpoints:
        status = endpoint.get("status") or {}
        host = (status.get("hosts") or {}).get("host")
        if status.get("endpoint_type") == "ENDPOINT_TYPE_READ_WRITE" and host:
            return endpoint["name"], host
    return None


branch_name = wait_for("the default branch", default_branch, timeout_s=WAIT_LAKEBASE_S)
endpoint_name, pg_host = wait_for(
    "the read-write endpoint", read_write_endpoint, timeout_s=WAIT_LAKEBASE_S
)
databases = list_all(f"/api/2.0/postgres/{branch_name}/databases", "databases")
matching = [
    d for d in databases if (d.get("status") or {}).get("postgres_database") == LAKEBASE_DATABASE
]
if len(matching) != 1:
    found = [(d.get("status") or {}).get("postgres_database") for d in databases]
    raise RuntimeError(
        f"Expected one database named {LAKEBASE_DATABASE!r} on {branch_name}; found {found}."
    )
database_name = matching[0]["name"]
print(f"Branch:   {branch_name}")
print(f"Endpoint: {endpoint_name}")
print(f"Database: {database_name}")

## 6. Create the App, with user authorization and the Lakebase resource

`user_api_scopes` is what turns user authorization on for this App; without it the edge has no
visitor token and group grants cannot be resolved. Attaching the Lakebase database as a
resource is what makes the platform create the App's service principal role in Postgres and
set `PGHOST` for it. If the App exists already, it is updated with the same full body (an
update replaces the whole body and takes no default scopes).

The size is not part of that body: the update call refuses a `compute_size` (even the current
one) and answers "Compute size updates are not supported in this update API". The size is set
afterwards through the separate app-update call, only when it differs from what the App has.

In [ ]:
app_body = {
    "name": APP_NAME,
    "description": "appmux: one App slot that hosts many child apps",
    "user_api_scopes": USER_API_SCOPES,
    "resources": [
        {
            "name": "lakebase",
            "description": "The registry database",
            "postgres": {
                "branch": branch_name,
                "database": database_name,
                "permission": "CAN_CONNECT_AND_CREATE",
            },
        }
    ],
}
existing_app = get_or_none(f"/api/2.0/apps/{APP_NAME}")
scopes_before = set((existing_app or {}).get("effective_user_api_scopes") or [])
if existing_app is None:
    api("POST", "/api/2.0/apps", body=app_body)
    print(f"Creating App {APP_NAME}; waiting for its compute to start")
else:
    api("PATCH", f"/api/2.0/apps/{APP_NAME}", body=app_body)
    print(f"App {APP_NAME} exists; updated it with the full body")


def compute_is(state):
    def check():
        current = api("GET", f"/api/2.0/apps/{APP_NAME}")
        compute = (current.get("compute_status") or {}).get("state")
        if compute == "ERROR":
            raise RuntimeError(f"The App's compute is in ERROR: {current['compute_status']}")
        return current if compute == state else None

    return check


app = wait_for("the App's compute to be ACTIVE", compute_is("ACTIVE"), timeout_s=WAIT_APP_S)

if COMPUTE_SIZE and app.get("compute_size") != COMPUTE_SIZE:
    print(f"Changing the compute size from {app.get('compute_size')} to {COMPUTE_SIZE}")
    api(
        "POST",
        f"/api/2.0/apps/{APP_NAME}/update",
        body={
            "update_mask": "compute_size",
            "app": {"name": APP_NAME, "compute_size": COMPUTE_SIZE},
        },
    )

    def size_is_applied():
        update_state = (api("GET", f"/api/2.0/apps/{APP_NAME}/update").get("status") or {}).get(
            "state"
        )
        if update_state == "FAILED":
            raise RuntimeError("The platform could not change the compute size")
        current = api("GET", f"/api/2.0/apps/{APP_NAME}")
        compute = (current.get("compute_status") or {}).get("state")
        done = (
            update_state == "SUCCEEDED"
            and current.get("compute_size") == COMPUTE_SIZE
            and compute == "ACTIVE"
        )
        return current if done else None

    app = wait_for(
        f"the App to run on {COMPUTE_SIZE} compute", size_is_applied, timeout_s=WAIT_APP_S
    )
print(f"Compute size: {app.get('compute_size')}")

sp_client_id = app["service_principal_client_id"]
print(f"Service principal: {sp_client_id[:8]}... (kept in sp_client_id)")

in_effect = set(app.get("effective_user_api_scopes") or [])
refused = sorted(set(USER_API_SCOPES) - in_effect)
print(f"Scopes in effect: {len(in_effect)} of {len(USER_API_SCOPES)} asked for")
if refused:
    print(
        "NOT in effect (a workspace admin may need to allow them, or the feature is off): "
        + ", ".join(refused)
    )

## 7. Permissions

Two kinds, both attached to the App or to its service principal, so both are lost with an App
that is deleted: run this section again after you create the App anew.

**7a. Who may open appmux.** `APP_ACL` is the outer layer; appmux's own grants decide which
child each person reaches. This adds to the App's permissions and removes nothing.

In [ ]:
if APP_ACL:
    api("PATCH", f"/api/2.0/permissions/apps/{APP_NAME}", body={"access_control_list": APP_ACL})
    for entry in APP_ACL:
        principal = (
            entry.get("user_name") or entry.get("group_name") or entry["service_principal_name"]
        )
        print(f"{entry['permission_level']} on the App: {principal}")
else:
    print("APP_ACL is empty: only you and workspace admins can open the App.")

**7b. What appmux's service principal may use.** appmux reads its content store and a child's
secret references with the App's service principal (a shared-service-principal child also runs
as it). Nothing in appmux grants those: this cell gives the new service principal what you
listed in `CONTENT_STORE_FOLDER` and `SP_GRANTS`. It must run **before the deploy** (cell 11),
because appmux creates its store folder at its first start and stops if it cannot.

What appmux itself needs (the Lakebase project and the Postgres roles) is made by cells 5 and 9, so
`SP_GRANTS` lists only what your child apps read.

A grant can fail (you need the right to grant it). Every entry is tried, each result is
printed, and the cell stops at the end if any failed; fix the entry and run it again.

In [ ]:
SECRET_ORDER = ["READ", "WRITE", "MANAGE"]


def grant_secret_scope(entry):
    wanted = entry.get("permission", "READ")
    if wanted not in SECRET_ORDER:
        raise ValueError(f"permission must be one of {SECRET_ORDER}")
    current = get_or_none(
        "/api/2.0/secrets/acls/get", {"scope": entry["scope"], "principal": sp_client_id}
    )
    if current and SECRET_ORDER.index(current["permission"]) >= SECRET_ORDER.index(wanted):
        return f"already has {current['permission']}"
    api(
        "POST",
        "/api/2.0/secrets/acls/put",
        body={"scope": entry["scope"], "principal": sp_client_id, "permission": wanted},
    )
    return f"{wanted} granted"


def grant_workspace_folder(entry):
    level = entry.get("permission_level", "CAN_READ")
    status = api("GET", "/api/2.0/workspace/get-status", query={"path": entry["path"]})
    if status.get("object_type") != "DIRECTORY":
        raise ValueError("this path is not a folder")
    api(
        "PATCH",
        f"/api/2.0/permissions/directories/{status['object_id']}",
        body={
            "access_control_list": [
                {"service_principal_name": sp_client_id, "permission_level": level}
            ]
        },
    )
    return f"{level} granted"


def grant_unity_catalog(entry):
    api(
        "PATCH",
        f"/api/2.1/unity-catalog/permissions/{entry['securable_type']}/{entry['full_name']}",
        body={"changes": [{"principal": sp_client_id, "add": list(entry["privileges"])}]},
    )
    return f"{', '.join(entry['privileges'])} granted"


def grant_permission(entry):
    api(
        "PATCH",
        f"/api/2.0/permissions/{entry['object_type']}/{entry['object_id']}",
        body={
            "access_control_list": [
                {
                    "service_principal_name": sp_client_id,
                    "permission_level": entry["permission_level"],
                }
            ]
        },
    )
    return f"{entry['permission_level']} granted"


GRANTERS = {
    "secret_scope": grant_secret_scope,
    "workspace_folder": grant_workspace_folder,
    "unity_catalog": grant_unity_catalog,
    "permission": grant_permission,
}

wanted_grants = []
if store_folder:
    # Creating a folder that exists is not an error; the new folder is yours to manage.
    api("POST", "/api/2.0/workspace/mkdirs", body={"path": store_folder})
    wanted_grants.append(
        {"type": "workspace_folder", "path": store_folder, "permission_level": "CAN_MANAGE"}
    )
wanted_grants += SP_GRANTS

failures = []
for entry in wanted_grants:
    label = " ".join(
        str(v)
        for k, v in entry.items()
        if k != "type" and not k.startswith("permission") and k != "privileges"
    )
    try:
        print(f"ok      {entry['type']} {label}: {GRANTERS[entry['type']](entry)}")
    except Exception as error:  # report every entry, then stop below
        failures.append(entry)
        print(f"FAILED  {entry['type']} {label}: {type(error).__name__}: {str(error)[:300]}")
if not wanted_grants:
    print("Nothing to grant: CONTENT_STORE_FOLDER and SP_GRANTS are empty.")
if failures:
    raise RuntimeError(
        f"{len(failures)} of {len(wanted_grants)} grants failed (listed above). "
        "Fix them in cell 1 and run this cell again before the deploy."
    )

## 8. Database driver

Installs `psycopg` from the zip's own `wheels/` folder, checking the hashes in its
`requirements.txt`, into a scratch folder (`--no-deps`: the zip holds only what the Apps
environment lacks, and a notebook environment already has `typing-extensions`). If the wheel
does not fit this Python, it falls back to the same pinned version from the package index.

In [ ]:
if subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True).returncode:
    raise RuntimeError("This compute's Python has no pip module; use a cluster that has pip.")
deps_dir = tempfile.mkdtemp(prefix="appmux-deps-")
pip = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--disable-pip-version-check",
    "--no-input",
]
offline = subprocess.run(
    [
        *pip,
        "--no-deps",
        "--target",
        deps_dir,
        "--no-index",
        "--find-links",
        str(root / "wheels"),
        "-r",
        str(root / "requirements.txt"),
    ],
    capture_output=True,
    text=True,
)
if offline.returncode != 0:
    pinned = re.search(r"^psycopg==([\w.]+)", (root / "requirements.txt").read_text(), re.M)
    print("The zip's wheels do not fit this Python; trying the package index.")
    print(offline.stderr.strip()[-400:])
    if not pinned:
        raise RuntimeError("requirements.txt pins no psycopg version.")
    online = subprocess.run(
        [*pip, "--no-deps", "--target", deps_dir, f"psycopg[binary]=={pinned.group(1)}"],
        capture_output=True,
        text=True,
    )
    if online.returncode != 0:
        raise RuntimeError("Could not install psycopg: " + online.stderr.strip()[-400:])

sys.path.insert(0, deps_dir)
psycopg = importlib.import_module("psycopg")
sql = importlib.import_module("psycopg.sql")

try:
    import pydantic  # the schema loader needs it; notebook environments ship it
except ImportError as error:
    raise RuntimeError(
        "pydantic is missing: run %pip install pydantic in a cell, then start over."
    ) from error
print(f"psycopg {psycopg.__version__} and pydantic {pydantic.VERSION} are ready")

## 9. Postgres roles

The grants of `migrations/0002_grants.sql` name two group roles, which you create before the
schema is loaded:

- `appmux_sp` holds the App's service principal role, so the grants name one fixed role
  whatever the client id is;
- `appmux_platform_writers` holds the role of each platform admin who will write platform data
  with their own login.

The service principal role is created by the platform once the App has its database resource;
if it has not appeared after three minutes it is created here (as an OAuth role, the way the
Lakebase CLI documents). Each admin needs a Lakebase role of their own; yours exists because
you own the project.

In [ ]:
# The two group roles the grants of migrations/0002_grants.sql name.
SP_GROUP = "appmux_sp"
WRITERS_GROUP = "appmux_platform_writers"
POSTGRES_PORT = 5432
# A Lakebase role reaches Postgres a little after the control plane lists it.
ROLE_WAIT_S = 180


def database_token():
    """A one-hour OAuth token for the person running this notebook. Never printed."""
    return api("POST", "/api/2.0/postgres/credentials", body={"endpoint": endpoint_name})["token"]


def lakebase_roles():
    listed = list_all(f"/api/2.0/postgres/{branch_name}/roles", "roles")
    return {
        ((r.get("status") or {}).get("postgres_role") or (r.get("spec") or {}).get("postgres_role"))
        for r in listed
    }


def ensure_lakebase_role(postgres_role, identity_type, role_id):
    if postgres_role in lakebase_roles():
        return
    operation = api(
        "POST",
        f"/api/2.0/postgres/{branch_name}/roles",
        query={"role_id": role_id},
        body={
            "spec": {
                "identity_type": identity_type,
                "postgres_role": postgres_role,
                "auth_method": "LAKEBASE_OAUTH_V1",
            }
        },
    )
    wait_operation(operation, f"creating the Lakebase role {postgres_role}")
    print(f"Created the Lakebase role {postgres_role}")


# The platform creates the service principal's role by itself; give it a moment first.
try:
    wait_for(
        "the App's service principal role",
        lambda: True if sp_client_id in lakebase_roles() else None,
        timeout_s=ROLE_WAIT_S,
        every_s=15,
    )
except TimeoutError:
    ensure_lakebase_role(sp_client_id, "SERVICE_PRINCIPAL", sp_client_id)

for admin in platform_admins:
    role_id = "user-" + re.sub(r"[^a-z0-9]+", "-", admin).strip("-")
    ensure_lakebase_role(admin, "USER", role_id[:63])

# The group roles and the memberships are SQL, run with your own login.
with psycopg.connect(
    host=pg_host,
    port=POSTGRES_PORT,
    dbname=LAKEBASE_DATABASE,
    user=me,
    password=database_token(),
    sslmode="require",
    connect_timeout=30,
    autocommit=True,
) as connection:
    for group in (SP_GROUP, WRITERS_GROUP):
        exists = connection.execute(
            "SELECT 1 FROM pg_roles WHERE rolname = %s", (group,)
        ).fetchone()
        if not exists:
            connection.execute(sql.SQL("CREATE ROLE {} NOLOGIN").format(sql.Identifier(group)))
            print(f"Created the group role {group}")

    def has_postgres_role(name):
        found = connection.execute("SELECT 1 FROM pg_roles WHERE rolname = %s", (name,)).fetchone()
        return True if found else None

    # The Lakebase role reaches Postgres a moment after the control plane lists it.
    for member, group in [
        (sp_client_id, SP_GROUP),
        *[(a, WRITERS_GROUP) for a in platform_admins],
    ]:
        wait_for(
            f"the Postgres role {member}",
            lambda member=member: has_postgres_role(member),
            timeout_s=ROLE_WAIT_S,
            every_s=10,
        )
        connection.execute(
            sql.SQL("GRANT {} TO {}").format(sql.Identifier(group), sql.Identifier(member))
        )
        print(f"{group} <- {member if member != sp_client_id else 'the App service principal'}")

## 10. Load the schema

Runs `python -m appmux.registry.load_schema` from the unpacked release with **your** database
login, in a separate process, with the host, user and token in that process's environment
only. The App never runs this: its role must not own the tables or hold DDL rights.

| Exit | Meaning |
|------|---------|
| 0 | The registry was empty and is loaded. |
| 3 | Already loaded; nothing was changed. This is not a failure. |
| 2 | Refused before anything ran (a setting or a group role is missing). |
| 1 | Failed; nothing was committed. |

In [ ]:
LOADER_TIMEOUT_S = 300
LOADER_DONE_EXITS = (0, 3)  # 0: the schema was loaded now; 3: it was already loaded

token = database_token()
environment = {key: os.environ[key] for key in ("PATH", "HOME", "LANG") if key in os.environ}
environment.update(
    PGHOST=pg_host,
    PGUSER=me,
    PGPASSWORD=token,
    PGDATABASE=LAKEBASE_DATABASE,
    PYTHONPATH=os.pathsep.join([deps_dir, str(root)]),
    PYTHONDONTWRITEBYTECODE="1",  # keep __pycache__ out of the folder the App deploys from
)
loader = subprocess.run(
    [sys.executable, "-m", "appmux.registry.load_schema", "--migrations", str(root / "migrations")],
    cwd=root,
    env=environment,
    capture_output=True,
    text=True,
    timeout=LOADER_TIMEOUT_S,
)
print((loader.stdout + loader.stderr).replace(token, "***").strip())
del token, environment

if loader.returncode in LOADER_DONE_EXITS:
    print(f"\nSchema step done (exit {loader.returncode}).")
else:
    raise RuntimeError(
        f"The schema loader exited with {loader.returncode}; read its message above."
    )

## 11. Deploy

Deploys the folder as a snapshot. The first start answers "starting" at once, then appmux runs
its `self_check()` and the status page shows the result.

In [ ]:
deployment = api(
    "POST",
    f"/api/2.0/apps/{APP_NAME}/deployments",
    body={"source_code_path": str(root), "mode": "SNAPSHOT"},
)
deployment_id = deployment["deployment_id"]
print(f"Deployment {deployment_id} started")


def deployment_finished():
    current = api("GET", f"/api/2.0/apps/{APP_NAME}/deployments/{deployment_id}")
    state = (current.get("status") or {}).get("state")
    return current if state in ("SUCCEEDED", "FAILED", "CANCELLED") else None


finished = wait_for("the deployment", deployment_finished, timeout_s=WAIT_APP_S)
state = finished["status"]["state"]
print(f"Deployment {state}: {finished['status'].get('message', '')}")
if state != "SUCCEEDED":
    raise RuntimeError("The deployment did not succeed; read the App's logs in the Apps UI.")

## 12. Restart if the scopes changed

A scope change reaches tokens only after a stop and a start. A new App already started with
its scopes, so this cell does nothing for it.

In [ ]:
if existing_app is not None and set(USER_API_SCOPES) != scopes_before:
    api("POST", f"/api/2.0/apps/{APP_NAME}/stop")
    wait_for("the App to stop", compute_is("STOPPED"), timeout_s=WAIT_APP_S)
    api("POST", f"/api/2.0/apps/{APP_NAME}/start")
    wait_for("the App to start", compute_is("ACTIVE"), timeout_s=WAIT_APP_S)
    print("Stopped and started the App so the new scopes reach tokens.")
else:
    print("The scopes did not change; no restart needed.")

## 13. Status

In [ ]:
def app_is_up():
    current = api("GET", f"/api/2.0/apps/{APP_NAME}")
    state = (current.get("app_status") or {}).get("state")
    return current if state in ("RUNNING", "CRASHED", "UNAVAILABLE") else None


final = wait_for("the App to report a state", app_is_up, timeout_s=WAIT_APP_S)
print(f"App:     {(final.get('app_status') or {}).get('state')}")
print(f"Compute: {(final.get('compute_status') or {}).get('state')}")
print(f"URL:     {final.get('url')}")

### What to check next

1. Open the URL as one of the platform admins. The status page shows the `self_check()` result;
   if the App is `UNAVAILABLE`, read its logs in the Apps UI first.
2. The first admin signs in; `platform_admins` is then set from `APPMUX_BOOTSTRAP_ADMINS`
   (it is honoured only while that setting is unset).
3. To add a platform admin later: add the address in the admin UI **and** put the address in
   `MORE_PLATFORM_ADMIN_EMAILS`, then run cells 1, 2, 5, 6 and 9 again (cell 9 gives their
   Lakebase role and membership of `appmux_platform_writers`).
4. To upgrade: upload the new zip, set `ZIP_PATH` and `REPLACE_EXISTING_FOLDER = True`, and run
   cells 1 to 4, 5, 6, 8, 10 and 11. Cell 10 answers 3 ("already loaded"); a migration file
   added after your first load is applied by hand
   ([TD-008](https://github.com/timezlab/databricks-appmux/blob/main/docs/tech-debt.md)).
5. If you delete the App and create it again, its service principal is a new one. Keep the
   same `CONTENT_STORE_FOLDER` and the same Lakebase project, then run the notebook from cell
   1 to the end: cell 7a gives the App its permissions again, cell 7b gives the new service
   principal its folder and `SP_GRANTS`, and cell 9 puts it into `appmux_sp`. The children
   and releases are still in the registry and the store, but a child that reads a secret
   the new service principal cannot read fails to start until that scope is in `SP_GRANTS`.
   Without `CONTENT_STORE_FOLDER` the old service principal's folder is not readable by the
   new one and every release must be built again.